# CDCN CVPR 2020 · Kiểm tra MTCNN và PRNet trên OULU-NPU
Dùng runtime **CPU** và chạy các cell theo thứ tự. Mẫu nhỏ lấy 3 video và
9 khung ở mỗi tập Train và Dev của Protocol 1. Mỗi tập có video thật,
ảnh in và phát lại. Kết quả là ảnh scene, bbox, depth JPEG và JSON kiểm tra.
Notebook dành cho bước chuẩn bị dữ liệu trước huấn luyện.

In [ ]:
#@title 1. Kết nối Drive và chọn thư mục OULU-NPU
from pathlib import Path
import os
import shutil
import sys
import time
from google.colab import drive

drive.mount("/content/drive")
DATA = Path("/content/drive/MyDrive/face-pad/datasets/OULU-NPU")
WORK = Path("/content/cdcn-prnet-smoke")
RUN = Path("/content/drive/MyDrive/face-pad/cdcn-prnet-smoke-p1") / str(time.time_ns())
for name, size in {"Protocols.tar": 417792, "Train_files.tar": 37753662976,
                   "Dev_files.tar": 28151992320}.items():
    source = DATA / name
    if not source.is_file() or source.stat().st_size != size:
        raise ValueError(f"Kiểm tra đường dẫn và dung lượng: {source}")
RUN.mkdir(parents=True)
print("Thư mục kết quả:", RUN)

In [ ]:
#@title 2. Nạp code và cài môi trường tách riêng
BUNDLE_SHA256 = 'e8d6de9addd53021af25d75b9fea5e0fb67fa8b380003eb9ada9cefef7a53fe1'
BUNDLE = (
    'H4sIAAAAAAAC/+08a1PjSJLz2RH7H+o0H1bqloUfQNPEeCMYoHu46QcHTN9tMIRClkpYjSzpJBnweNnfvvmo0sMWML23MREbhyOm'
    'kUuVWVn5zqzyFLm/FUiZhZ4v3cwLtlw3SqLSdZ1s+d2/6DOAz+72Nv2Fz/rfweDNm/oZx4fj0Xj4nRh89wd8FkXp5bD8d/8/P4Zh'
    'HIH434H4xenBkchlIb3cn4nM82+8a+nAhD/1/tRz3VuZF1GauK6YCGPgDJ0BvPju5fPv/SnW7T/L02tQguKPs/+dneF43f7H490X'
    '+/+D7P9U5lEaRL7QohdhmoskLeU0TW/E13RaCC8JRJymWYH+oBfm6Vy4brgoF7kEhxDNszQvYRIAeSU4iaLHczKvnMXRVE84ha89'
    '9VwsprCeD8vpkXKWSy+IkutqIJrLXq/nxx7QdKqI2+8J+AQyFDpSmYWMQ1vE3lTGtiiBhHjyKU2kLRbwfmJEpZwXhi2ipJT5Lbwc'
    'DyzGgh8EdhQsPRMC9YwI1KOGBu/XXIkXqZG38QZABgAM1lcrSneRBV6JL2kIlbCUAXzFTTvzFDiZJpFvWm3QokwznKRZ5RzfyqSE'
    'WRVXGK/iCa7f2GsU0oj4j0lNXv12nWr8s/myTfwGtRUdc5AVxA9FCOyvbFKSpHdP7VXGXlYQO3Biv8WjapIisyK5euGni6SEN6Eh'
    'VrRF+2FrVcsWvopVJd4HA9lSvxVRgbovUIOAjkIKUPiKrtLD6Gc0OUpbExMYzhdJAjIxyFoaGPHrQPzABP/QeNPmfRTEsr3jBq9b'
    'M5kME/b3N3F8cSD+vjIby/VZ7OJVxcYtGtl3BuFDQdulpX5YU+xXYtRapZIF8ICWSlKRyDvFXvQRK8Sj0NbCyyW4hQRALle1cT1c'
    'Ac+RUw8rgn8AfJq8lXpgTCvY3YNRK5I7g4SknEqvpE02lOhuFsE2UFiVbTh3XsTzqn1ZbS5nObzgGVpFK8FZtgjjRTGbXOQL2dBl'
    '15WITLmaBsIuZEpN15G1jIjtt2XIF/RkAvC1LCc0q945WLInwVAex8XWYW5IAae0NnLfcJny3nfLZSb5CXi1gMcyh1RgCtnfupMk'
    '/hayNLsJ+JpGSeNVF2tCDwQWkAbqpZWJheDKi9km13o9JBvk48bp9bUMTD+dz8GebPGq8vjwxhb+XaCcvkxu6UmRD/HqXZrfeXkA'
    'ijufRgkonA+KE4h0UWYLiDIpSEEKX8ZsqRmmwPmtpNE0j66jBIxD5nmaU/BDpLAkCA/DmQmPlh5zMi8HTXHmN0GUm/yloJ0ghyMw'
    '5vSmIUK1F8B0WZS5CZK3yKzgAeKJfn3Vq9kZGh/S632xgrUejE39uovKGdGRZjIxjXsDueGnqF8TY1GG/T3DEl6BO7ergGoqNlbh'
    'cbcZHgllHaudU0JdSQHZDv8x1+E/9PQBoJ80QU5Oj+0uz9IMLWUAHG5CnV8cff7lAtRR3peahWt7sVkqxcTIZRaD1vLuEEPb5st8'
    'ub9BAHI6BnVAViOIw5Tvd1IKL5y7HDIJE0GsR+eQRMzu9yxChMetBBPD6PYQdRgLMB4QbeTW2lPAgmRWih+9Qh7TI+Rdm8SDqRGC'
    'LI1j08Lg9mkj6NcEwkRQgjloPOx0k6JOPlaQRCOGdJT/zia0Irgh4wuefHyfRTm4/8dR30RI/TNLb77PvaiQzViNLG0vQ1PEGUQk'
    'oOUY1QmMbMUhC20WkzKyAZLGCv99cMS7BTgL9h/aGq1/z1q48PMoK4stP/ATN8sTWbrFPL2R/8Ly75n6bzwc7e5s1n+jl/rvD6r/'
    'zqQPFVuZL/xSHB4dfuoX5RLyqo8Xh58+bZ2efZIluEiMlFCGyZKrQ09MIYsLwDhIWyARKH93ZajH8muIkGCf6rtf3OrHmVdg2ai/'
    '8p+uAUgtSg/SY48XBmvNyzSNC72uP/OiREN9LdLqed6oRNPi0WIVoiTs4zQtovtW7TpblFH8eCVbLOui1stDSHqqr3Ketb5jiauf'
    'IR2q2ePfjvRjsphnS4xsScaUFn6ULZ0CCI6wYOFZ/zUDp0QeTBXfzbaOU9X2enfqe68HEj6+cA8/f/x4coHFTegPR6HckcNwuL07'
    'mgbS2xltD95OR8NxMPDDob/nh7v+ztToHR2fXvzUgPR3g+EgHMnprjd9uzvypuPhdrg3lN625w0GXjgM3o6D6RQg//v45P1PF+75'
    'TwcIt/M2GI/3puHAHw2nw8FU7g3H4fBNuLc92B758NLbmYbj4Z7c2dvb29kJw9F05A+2h/5oZzp9O/SM3smno+P/0ej86d7e8O0o'
    '9MM329vTPTkCsqS3641294Z+8HYQDgcjbxS8ffPGmw4h0R0PRrvTINz25Zu9HQAxeucfP/987J4eHP588P74HEstTiVLmRRpHsbp'
    '3WQycoa7zsiw1ZuwfyNzr1Djg8Z4EUfzyWRIDVM1Oi/9JMGxQT1GQsax0a6zrQdB0DdR2Y/mkD5PJgNntF0DkBIQ4jfOUA9idubf'
    '9rNlOUuTPtQPUKAVQNW2M8TV9nZhoqX2d/7h8wVtbmCLsS3eWL3TExiqxstFFksz9xJI3fcsnYsXM89ES1E5IsVG/K5yznzKSRj4'
    'E+nN99fLEWXZDtqAG0TXsoAagabasKOZN9rZNSxnJu/VO71qnHqBO4fgGxcmRSkbMIL/yWWuCEkLB/LPKE+TS+PinfvL+bH74fj9'
    'weFf3Z+Pzw7OjStUjaGxMffwl6MD98vJ+cmPH47do+MvJ4fHajLPrQxXi96B5Bc27NwOcZ9lSLPI4EisGoDcJ1de34uLdwL14s+F'
    'YOBoGsVRuaRtyRzyjEz64Fmp4oDKRPXe4bVPTtMhNOBVHGDBAiR6CYrlksI5sbz2/KUL32NvKfOCaK8dJD+5DGfWYEXub4Ja37CO'
    'k6T53Iuj34jCJ1etlODp5dcwEpSlStcS+ANhZ8JsNQN5G/lyYhye/rI/0GSDouP5xHXuZTNU39B5j49mozZqzXG8wgXd8hYx6Fmt'
    'qd+LDyAVkkUWJVguQkUYRLQ+IfFAnMN+lPQv3o3W5Mn7duoiGAyDUnjQWLEljAoTpFdGbz3RRyvCUtrFmsfcqNwcVeWYrQTW2FBQ'
    'Vstfky7NTcocxMP8pnl+/GtCytsxLcy9ubxL8xtRh2y38MHQDfsZGtaNpKYmdNEhrq2tBp9Ypt49ekqXc+9iCxywaA3ASLMHoQU3'
    '4ejqfCSNvIBnsxZGA6Kl+/WEKxKi+taaTEKLkgISD/SjWMeTtK0a50bVJO+ljyV0Bk7QROErOPSqsB18jX2QWllcFx9ct13hEOu8'
    'LKpC+tmn1ntyl9imOPtkRqDqYJiTd15cSEIdRveTLmpDbHfEaxRXO83SzBxYG+8Uy+h1g202FZs8nd11pyACKAtnLgU5JQplN9rF'
    'o+lgxlVsNaZWJtRiZ21E02UpC9Pq5i1jXmOsilHa29jMwWrytUzAd5UQt2oidIOqTqFNshr7MSzK0eTXU9gf5HiOf1sepjGWnQyH'
    'Y4efP3w+c398fzY6e/8j04aJXIFNeYXW4QeXxk1Ax9OgxsV+KI02gi8VuV+wv8clrkFuVARRQNMZFzi2kJooBHdvi6Ut7myBgph7'
    '9ybhtMWNXE5ibz4NPEy35/v076UxTe+Nq8vRlXjVGhhfWeqR5QqZNFUBiBPZ4qjMGXdgQ37reHnuLc1LWP1evMblgYglPM2ulI7e'
    'gqVFzAvGcA0lqx40qwV04MgaimTqaXYDNPaSYO7lN01YG+XjQDqCrdGo+CiDd1EMtUWjUaN0xazZZNm8nFIJP08zl4qkwgQG2EKh'
    'K3wvlpC1jZUmtPgME2lwJqPrGeQ4d1FA9BPs5f7oSuUTP0NqTxGKEwjIK8Bb5hQ3RZqjnXmgBPCyhADSz9IoKSGY3YM7IIq8PAIT'
    '5B4kd7aBCvW0xIwQeU//3FkWGN/IFiYKgf6Z8RDT6cWhi7TjXyIUdwc6cMdQ+uusgohlWCqFAneJZZgDdIJKakJEX2FV8s6REwgR'
    'JSaxoxvqdRuKj8keXWWpV5mp+dO0LMGb8jKa+11wr9twYG+0o79MFKEQa3Dtv0wUyqeMEGpByBmIRwGZHsrfaOkXhEUIeYDRVvjQ'
    'n9EYLmvzolVqLmN0CbdRIFNMk9My9VM8ysziiJVJuoU3h6ye40AzhVd1KmfxNaiR73M+j3dColvZ3A6e4KlhBxKW3PNLRGGGxqmC'
    'L7b0kzvcWhEZD055X1Iu4wWmBV4M45/JKU6/iK719tM7tPBLk1qm2I2DjSfgIB3wslFmWtwpV61rfEEd8+KWEIOpA3kOLYj91gLn'
    'o6hwusagbelzIgWxjDBKD+++QPEkt1SyqI4N2LiwXU9TmNd4loCn02xLBOYqMCZ/aIshUG5u22JkXSENDSnw2YeaxOT4s1Q5t0uT'
    '0Nk6Z7WF56On5I2331ELu704NWkIAF/iEmCO22oV1hBYZPXQ063w7tWIq0xTLfoOzqO82ilDGMk4oI2gCDEmMOX4hBAsTJSQabiG'
    'IqzRqjUZweXgyla4Lof1I8QVPHJ9jEedfXDWIwQbWngEbqot0kB383nDZkODkcyjAryDP9sXK9zIg7HZfGYmX+JrTB95sY1ZU9DX'
    'm17zrLWrP92iANYuwK3rzhsYXrmAOKzsS6yIJ3TETUyBJ177oe1amD7lOuQ8Uo6DUiXNyMp3cHqt4iYOx2lZTBp9BDqQKzIuX4Do'
    'pOVifC9T6c4XXOPQy7A/aFaZmXLazURZH+Kj9gA4xmqTcqODU/f07POp++7s4OOxe/j5l08XjfSVmvwI+QPIG1wZEUrG3ySOLW84'
    'sJ5ltk+tS8qQ8FxECkrTeA3gOaWb6xqAxmCLBV9guLxC9WyeTnwvjuQt5FEy44o/W0zBo9ehHP1TIB1xQZQCAUK5Cw9yioJh7mYp'
    'DBx8OXFaVom7ReOiXbf3lqRzOsgEZYcgRhNfKVZtbYlnWVXDmBXQcL0SUFEXQqhe7bXYgQDMEK/BzNatvLnSpgkqhLg4UUcousne'
    'PD0DVx3hzQn0QdQHVq0sRZpN2DHVo9GCYiySTouu0209d4Cp9KexAKQYQ1unIPXYDsaD/tBaO8zbMPyNPXRRv3bKGZHGtbVNqwbp'
    'rQsJP+uUwtvpKJtTUYu7fSN2CaJkIZ/DQGR9IwqCcbwgMDWmLgHzbYSWUzj9fM6OARxSRUR/XfGoB3ijZqB6ACpOSLo2g/af3mBm'
    '502Lbl9UL2vBahXNmAYOnJ1v3vwjp6xURtQlzUbJ+Yq9c/c5aZ5GuNP1qoRAudR59MTWrD2i3ThmsL5dL4h4Xg0jcGNxLG6QxTwj'
    'oKshMAMqwgVEgL3fGZ4N7qSCGUAtC3T2C0jcpSAUinNzLzO6tYEabjJw51WwiuY8ZhrO1+zaUMx/QkmeIvPzuaIRvErMdTdjV4T9'
    '5+nx+w7KgJrxSNGTyyL6TZpMmkqdG1Rz8+Dk49nxwZH7/uzgr+eHBx+OrUsQPQQhcwwZ4HhkPUo+reR4ydK0UBLN9IjyXpwD8sjK'
    'zKSp1rcqwPfiI+ZMFMDUWeJG/AO9xodCTCV4LSloy3g/qiM4yDld8uMgjF8f3JW2vf3BOHgwNoFQH1zVXlIXgLY4ycEmE/V0CgMe'
    'K5zwDEswdoImVegS0+No4eXjSEn6w6NutDW933i3qIu2/wMKpSKseXwRBpO3mjxLuRLr95hA2LABQgbJaoXpwXhiBzRbtfYamu+U'
    'qWr3WU+xECtdt1iEYXQPJg0BEApRxkgdd+PXxOBrbIAS94dF9z2Ujq8FvtpEjcHxmSiFuaDjZVBVB+bKoAwb5hn7tS7gwRcqCoyp'
    'ZNtgy4MBnYUbmOTgBPjzTCZiaFpgun7EHjnnDTCoM4hn0Ki7IIu5Ox4BFKbh7CBgCNgMKClpc/nCovubzNOKYPTcwwfrmWJH1SHI'
    'oV5n55nDciy9QlYHgXNM5JSS0fWBnLoQfJXAOcivF+Czy1N6YwaSr7dAFTRx3SD1daeXIVF0rqdATKPfx5sE4OUx+EzwvP/Jydyv'
    '/r2zsezvcx5vYHVFNBmQxOWgfGByMDiTcTYxoGpKdcJPlZzYruv8kfHkIiQR6vhFqC9PrgOJN3oAQb0u1pRCZMBO0tGn1yHdaG4d'
    'u9v/u8BLXA3/8Qiwbur/s/DsXp+FBpCCDtUICf1BNIVZNe7wwmWBXsDT1+K4p0DDJN1qvHHjl/FJDuXFIsviJfAQ4lguWIEwdmrt'
    'qFrWuiPUaLjcyCUSnqW2qLIWbNWYhmYvEaJOu5u3NLChZDTYSPP0d1s0r2U0YzRe2oywpG7cwPNn0r9Rh2fmpXEd4bpG/9DgYxMk'
    'D009l7d92jm+/QnSC+OqcTPT0i21dhVOq4En0Lt7ttZeAUseBJEEFIn5oighDRArjaAZGTRPLwEG2yu8HPczpWoZ1/zDIHwEstmC'
    'ZxLS1mhn18X/IL8I/QSfGIz0oU/30PppSA/q0kDEdx94GjWQGrdYsPauXq7FmQgEc29YBFJdVHmqL6wSIr56RezgDj6A9YFSEURh'
    'WB/X0y5Zgk9Hdm7I1C3f6gKwQQfefCSEiR4tbDQ0h+sK/OVL4xpGUztbGlgZWG1IjZ+gkF3p9JqKLnX3WU1t93JaZvj7mzV8WbNG'
    '2tScR6somtlRRanchysTSo+fIcRgyXF8w1okEdzqb9ce68lUQ5Cor7xexj9LWCs8+CqxwU7h5OPBe7ytdH5u2Lw7fMng2H7AYI0j'
    'kLtTQ8NN0gTDNU+xHvmdAlPfq/tZeHl/kXNvGqz/kju21H3BQ+bG1SEteh2OXApH3EhqtAwZ/ukfPlFTi9JodI0GtcTQBx3J26aC'
    'Vg3t9hFI7eCBn9WhhANLGlVnk+Y0mvI1B9TZhk7rm8gweaduq4snHoxxTTjVlH384dwN7eTgy4mYy/kUL1/AflbNFVQXsVsYbXut'
    'UIuLgzOyWJIkwGI/jPeOXpufqkvxBGyLXzcbDs2znyZNm+c/dnWD0bmQeO3Ay5dHEHnxSBrrRrzWAeNrBiLnrAiqOaUo3GhRMWuQ'
    'MxunTVUXkxs2rVuZJsMRAx8vbnlOVNARFRW4fAJBXtqJ0zvIF8lFG453Gxn1BCzXEm6rVhv5xsIX73HF+gcjyB88QSXs+M/GdL5l'
    '1HGyxpvg+3XpIvelzajV3bs7dfcO0t5HfsbAF1fBDWRLRJhOv5oaEQLZYlu8EsPBSP/paJRM8ccxnj7Eb5wbECV289iDeHfVtrTO'
    'g4RHyibYOpZNzSWtbmat+05OZx288RtDjcfs8R77OcxXYEGy/mu8pmI2KEA1aBK0/2gZpbCqn40gKU6wmGdFcz+Pl5Z0sJy0947a'
    'WXe8HyFXeenOmrMSiq1ZBIMby3QUbqxliySOkpuu31lou4AcYJ7eSrNaaHNu9fty9UPVhssSfSKmwmZ1Qa/51tVT8A9bzdcPOkz8'
    'TazWg01fxaF9Z4g/Znz6hznKpTzjDdrFbks2SrFbsqFHvKGEKQR8r4756qBhPJA28mE1nvKkSPIGv9TtUQrUBWTO+pR1MjGsy8GV'
    'iqnSRxTtm85X6nyQrpNBsVIrCacAht4B9hbUIzYnyIXgHlRSDmOcCbvqSu9+I1debzzoqZQm1wBVpmxjL0LdIqjeYrL9RHS3Nhep'
    'Izy2KdaCvuqwkDlEeCWZLGwDieasoY599zt+ieAoJpAN8JG3PiTX4A8biAtMQwPEu66Y+FNgVk2uHNc9HR0CY5tmDpGYHF67p9Xw'
    'OyxXzAUDKA8mo/oKi5blE0VJaJzTcXOU4E27WJaw+ZWG0+m1SkvxRyQu56Zavyg3JeMERbJq97P+a88If6OK/ML/zQjEYtdF3XZd'
    'g0nj7s/L/3Xk5fPyefm8fF4+L5+Xz8vnn//8A9D2YGIAUAAA'
)
import base64
import hashlib
import io
import tarfile
bundle = base64.b64decode(BUNDLE)
if hashlib.sha256(bundle).hexdigest() != BUNDLE_SHA256:
    raise ValueError("Code đi kèm không khớp SHA-256")
WORK.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(bundle), mode="r:gz") as archive:
    for member in archive:
        dest = (WORK / member.name).resolve()
        if not member.isfile() or not dest.is_relative_to(WORK):
            raise ValueError(f"Invalid bundle member: {member.name}")
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(archive.extractfile(member).read())
(RUN / f"source-{BUNDLE_SHA256[:12]}.tar.gz").write_bytes(bundle)
sys.path.insert(0, str(WORK / "src"))
from deepface_pad.progress import Progress, run_logged

def run(label, *args):
    run_logged(args, label=label, log=RUN / f"{label}-{time.time_ns()}.log",
               env={**os.environ, "PYTHONPATH": str(WORK / "src"), "TF_ENABLE_ONEDNN_OPTS": "0"})

run("install-uv", sys.executable, "-m", "pip", "install", "uv")
VENV = WORK / "venv"
PY = VENV / "bin/python"
if not PY.is_file():
    run("create-venv", sys.executable, "-m", "uv", "venv", "--python", "3.12.10", VENV)
run("install-prnet", sys.executable, "-m", "uv", "pip", "install", "--python", PY,
    *('tensorflow==2.16.2', 'tf-keras==2.16.0', 'tf-slim==1.1.0', 'mtcnn==1.0.0', 'numpy==1.26.4', 'scikit-image==0.24.0', 'scipy==1.17.1', 'opencv-python-headless==4.11.0.86'))

In [ ]:
#@title 3. Tải code và trọng số PRNet công khai
PRNET = WORK / "PRNet"
RENDERER = WORK / "PRNet-Depth-Generation"
for path, repo, commit in [
    (PRNET, "https://github.com/yfeng95/PRNet.git", "fc12fe5e1f1462bdea52409b213d0cf1c8cf6c5b"),
    (RENDERER, "https://github.com/clks-wzz/PRNet-Depth-Generation.git", "c6d10f2eb6ab962ab314f81ea4aa00af1d93dbbb"),
]:
    if not path.exists():
        run(f"clone-{path.name}", "git", "clone", repo, path)
    run(f"checkout-{path.name}", "git", "-C", path, "checkout", "--detach", commit)
WEIGHT = PRNET / "Data/net-data/256_256_resfcn256_weight.data-00000-of-00001"
def weight_matches(path):
    if not path.is_file():
        return False
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest() == "59d338bf0c21b10be813f17f84042c9d3a5bf318e588855ff2b2c041c25bb91a"

if not weight_matches(WEIGHT):
    partial = WEIGHT.with_suffix(".part")
    run("download-prnet", "curl", "-LfsS", "--retry", "2", "-o", partial,
        "https://drive.usercontent.google.com/download?id=1UoE-XuW1SDLUjZmJPkIZ1MLxvQFgmTFH&export=download&confirm=t")
    if not weight_matches(partial):
        raise ValueError("Tệp tải về không khớp SHA-256 của checkpoint PRNet")
    partial.replace(WEIGHT)
run("verify-prnet", PY, "-u", WORK / "scripts/cdcn_prnet_smoke.py",
    "--image", PRNET / "TestImages/0.jpg", "--prnet", PRNET, "--renderer", RENDERER,
    "--output", WORK / f"public-{time.time_ns()}")

In [ ]:
#@title 4. Dựng dữ liệu mẫu Train/Dev và lưu lên Drive
OUT = WORK / f"oulu-{time.time_ns()}"
WIDE_SAMPLE = False  # True để thêm phone 4/session 2, đọc sâu hơn trong TAR
run("oulu-prnet-smoke", PY, "-u", WORK / "scripts/cdcn_prnet_smoke.py",
    "--data", DATA, "--prnet", PRNET, "--renderer", RENDERER, "--output", OUT,
    *(["--wide-sample"] if WIDE_SAMPLE else []))
with Progress("Save smoke assets to Drive"):
    local_zip = Path(shutil.make_archive(str(OUT), "zip", OUT))
    temporary = RUN / (local_zip.name + ".part")
    shutil.copyfile(local_zip, temporary)
    temporary.replace(RUN / local_zip.name)
    shutil.copyfile(OUT / "smoke_summary.json", RUN / "smoke_summary.json")
print("CDCN_PRNET_SMOKE_READY", RUN)